In [ ]:
# =============================================================================
# jobs/38_economics_assumptions.py
# The assumptions behind the indicative expected annual damage in the app (utils/economics.py has the method and the defaults).
#
# OUTPUT  config.TBL_ECON_ASSUMPTIONS  item, scenario (low, central, high), value, unit, source, price_year, note, updated_at
#           prob_high, prob_medium, prob_low, prob_very_low   annual probability of each NaFRA2 band (filled in from the band ranges)
#           damage_per_household                              GBP per household per flood event: LEFT EMPTY for you to set
#
# THE DAMAGE FIGURES ARE YOURS TO SET. Update the three damage_per_household rows (low, central, high) with the value, the source (for example the
# Multi-Coloured Manual average residential damage) and the price year. The app shows no pound figure until all three are set.
#
#   UPDATE <table> SET value = 30000, source = 'MCM 2024 average residential', price_year = 2024, updated_at = current_timestamp()
#   WHERE item = 'damage_per_household' AND scenario = 'central'
#
# RE-RUNNING IS SAFE. Only rows that are missing are added; a value someone has set is never overwritten. The checks at the end print any
# inconsistency (a probability outside 0 to 1, a scenario out of order).
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import importlib

import pandas as pd
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

import config
import importlib as _il

_il.reload(config)
import utils.economics as _econ
importlib.reload(_econ)
from utils.economics import COLUMNS, check_assumptions, missing_rows
from utils.helpers import get_spark, table_exists, utc_now

spark = get_spark()
TABLE = config.TBL_ECON_ASSUMPTIONS
now = pd.Timestamp(utc_now())

In [ ]:
# =============================================================================
# 1. ADD THE MISSING ROWS
# =============================================================================

SCHEMA = StructType([
    StructField("item", StringType()), StructField("scenario", StringType()), StructField("value", DoubleType()), StructField("unit", StringType()),
    StructField("source", StringType()), StructField("price_year", StringType()), StructField("note", StringType()), StructField("updated_at", TimestampType())])

existing = spark.table(TABLE).toPandas() if table_exists(spark, TABLE) else None
new = missing_rows(existing, now)
print(f"{0 if existing is None else len(existing)} rows already in {TABLE}; adding {len(new)}.")
if len(new):
    rows = [(r.item, r.scenario, None if pd.isna(r.value) else float(r.value), r.unit, r.source, None if pd.isna(r.price_year) else str(r.price_year), r.note,
             pd.Timestamp(r.updated_at).to_pydatetime()) for r in new.itertuples(index=False)]
    spark.createDataFrame(rows, SCHEMA).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TABLE)
spark.sql(f"COMMENT ON TABLE {TABLE} IS 'Assumptions for the indicative expected annual damage in the app: the annual probability of each NaFRA2 band and the damage per household per flood event, each low, central and high. Probabilities come from the band ranges; the damage figures are set by hand. Edited by hand, never overwritten by job 38. Public.'")

In [ ]:
# =============================================================================
# 2. CHECKS: what is set, and whether it is consistent
# =============================================================================

t = spark.table(TABLE).toPandas()
print("The table now:")
display(t.sort_values(["item", "scenario"]).astype(str))
dm = t[(t["item"] == "damage_per_household")]
set_n = int(pd.to_numeric(dm["value"], errors="coerce").notna().sum())
print(f"\nDamage per household: {set_n} of 3 scenarios set." + ("" if set_n == 3 else " The app shows no pound figure until all three are set."))
problems = check_assumptions(t)
print("Checks:", "all fine" if not problems else "")
for p in problems:
    print("  PROBLEM:", p)

In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success")